In [2]:
%pip install tensorflow


In [4]:
# ============================================
# CELLULE 1 : IMPORTS DES BIBLIOTHÈQUES
# ============================================
# TensorFlow pour le LSTM, sklearn pour les métriques

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

print(f" TensorFlow {tf.__version__}")

In [5]:
# ============================================
# CELLULE 2 : CHARGEMENT DES DONNÉES
# ============================================
# Même source Delta Lake que le notebook Prophet
# Agrégation horaire identique (date_trunc + avg)

df = spark.read.format("delta").load(
    "abfss://curated@energybigdatastorage.dfs.core.windows.net/ml_ready_dataset/"
)

from pyspark.sql.functions import col, date_trunc, avg

hourly_df = df.withColumn(
    "ds",
    date_trunc("hour", col("tstp"))
).groupBy("ds").agg(
    avg("energy_kwh").alias("y")
).orderBy("ds").dropna()

print(f" Données chargées : {hourly_df.count()} lignes")
hourly_df.show(5, False)

In [6]:
# ============================================
# CELLULE 3 : CONVERSION EN PANDAS
# ============================================
# Conversion pour utilisation avec TensorFlow/Keras

pdf = hourly_df.toPandas()
pdf["ds"] = pd.to_datetime(pdf["ds"])
pdf = pdf.sort_values("ds").reset_index(drop=True)

print(f"Période : {pdf['ds'].min()} → {pdf['ds'].max()}")
print(f"Nombre de points : {len(pdf)}")
print(pdf.head())

In [7]:
# ============================================
# CELLULE 4 : SPLIT TRAIN / TEST
# ============================================
# Même split que Prophet : avant 2014-01-01 = train, après = test

train = pdf[pdf["ds"] < "2014-01-01"].copy()
test = pdf[pdf["ds"] >= "2014-01-01"].copy()

print(f"Train : {len(train)} points | {train['ds'].min()} → {train['ds'].max()}")
print(f"Test  : {len(test)} points | {test['ds'].min()} → {test['ds'].max()}")

In [9]:
# ============================================
# CELLULE 5 : NORMALISATION DES DONNÉES
# ============================================
# LSTM nécessite des données normalisées entre 0 et 1

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train["y"].values.reshape(-1, 1))

print(f" Données normalisées")
print(f"Min : {train_scaled.min():.4f} | Max : {train_scaled.max():.4f}")

In [10]:
# ============================================
# CELLULE 6 : CRÉATION DES SÉQUENCES TEMPORALES
# ============================================
# Fenêtre glissante de 24 heures pour le LSTM

SEQ_LENGTH = 24

def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i-seq_length:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled, SEQ_LENGTH)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

print(f" Séquences créées")
print(f"X_train shape : {X_train.shape}")
print(f"y_train shape : {y_train.shape}")

In [11]:
# ============================================
# CELLULE 7 : CONSTRUCTION DU MODÈLE LSTM
# ============================================
# Architecture : 3 couches LSTM avec Dropout

model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQ_LENGTH, 1)),
    Dropout(0.2),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')

print(" Modèle créé")
model.summary()

In [12]:
# ============================================
# CELLULE 8 : ENTRAÎNEMENT DU MODÈLE
# ============================================
# 50 epochs, validation sur 10% des données

print(" Entraînement LSTM en cours...")

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

print(" Entraînement terminé")

In [13]:
# ============================================
# CELLULE 9 : PRÉDICTION SUR LE TEST SET
# ============================================
# Prédiction séquentielle point par point

full_data = scaler.transform(pdf["y"].values.reshape(-1, 1))
test_start = len(train)

predictions = []

print(f"Prédiction sur {len(test)} points de test...")

for i in range(test_start, len(pdf)):
    seq = full_data[i-SEQ_LENGTH:i, 0]
    seq = np.reshape(seq, (1, SEQ_LENGTH, 1))
    pred = model.predict(seq, verbose=0)
    predictions.append(pred[0][0])

predictions = scaler.inverse_transform(
    np.array(predictions).reshape(-1, 1)
).flatten()

print(f" Prédictions terminées")

In [14]:
# ============================================
# CELLULE 10 : ÉVALUATION ET COMPARAISON
# ============================================
# Métriques MAE et RMSE comparées avec Prophet

y_true = test["y"].values

mae = mean_absolute_error(y_true, predictions)
rmse = np.sqrt(mean_squared_error(y_true, predictions))

print(f"\n{'='*50}")
print(f" COMPARAISON PROPHET vs LSTM")
print(f"{'='*50}")
print(f"{'Métrique':<<15} {'Prophet':<<15} {'LSTM':<<15}")
print(f"{'-'*45}")
print(f"{'MAE':<<15} {0.021248517202228256:<15.6f} {mae:<15.6f}")
print(f"{'RMSE':<<15} {0.02593587093822413:<15.6f} {rmse:<15.6f}")
print(f"{'='*50}")

In [15]:
# ============================================
# CELLULE 11 : SAUVEGARDE DES RÉSULTATS
# ============================================
# Stockage dans Delta Lake pour comparaison avec Prophet

result = test.copy()
result["yhat"] = predictions
result["yhat_lower"] = predictions * 0.9
result["yhat_upper"] = predictions * 1.1

result_spark = spark.createDataFrame(result)

result_spark.write.format("delta").mode("overwrite").save(
    "abfss://curated@energybigdatastorage.dfs.core.windows.net/lstm_predictions/"
)

print(" Sauvegardé dans : curated/lstm_predictions/")

In [16]:
# ============================================
# CELLULE 12 : VISUALISATION
# ============================================
# Graphiques : prédictions vs réel + courbe d'apprentissage

plt.figure(figsize=(14, 6))

sample = result.iloc[::24]

plt.plot(sample["ds"], sample["y"], label="Réel", color="black", linewidth=1)
plt.plot(sample["ds"], sample["yhat"], label="LSTM", color="blue", linewidth=1)
plt.title("LSTM - Prédictions vs Réel (échantillon journalier)")
plt.xlabel("Date")
plt.ylabel("Consommation (kWh)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title("Courbe d'apprentissage")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()